In [ ]:
import random
import datetime
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from seqeval.metrics import f1_score, precision_score, recall_score
import re

# ========== 1. Генератор ==========
CATEGORIES = [
    "Супермаркеты", "Рестораны", "Транспорт", "Связь", "Медицина",
    "Аптеки", "Кафе", "Развлечения", "Шопинг", "Красота",
    "Образование", "Домашние животные", "Дети", "Подарки", "Штрафы",
]
MCC_CODES = ["5411", "5812", "6011", "4814", "5912", "5944", "7997"]
DESCRIPTIONS = [
    "Пятёрочка", "Перекрёсток", "АЗС Лукойл", "Макдоналдс",
    "Яндекс.Такси", "МТС", "Аптека Апрель", "Кофе Хауз",
    "Хлебница", "Wildberries", "Ozon", "Золотое Яблоко",
    "Skillbox", "Четыре Лапы", "Детский Мир", "Flowwow",
    "Штраф ГИБДД", "Магнит 3", "Перевод на карту",
]
OPERATION_TYPE_LABELS = {
    "расход": ["Покупка", "Списание", "Оплата", "Платёж", "Платеж"],
    "доход": ["Зачисление", "Возврат", "Поступление"],
    "перевод": ["Перевод", "Перевод на карту", "P2P"],
}
NOISE_PREFIXES = [
    "", "Справка по операции ", "Детали транзакции: ",
    "Выписка ", "Операция по карте ", "Информация: ",
]
NOISE_SUFFIXES = [
    "", " Обработано банком.", " Статус: успешно", " PDF", " Копия.",
]
MID_NOISE = [" MCC ", " банк ", " *** ", " | ", "  "]
SEP_CHOICES = ["\t", " ", " | ", ""]


def _shift_entities(entities, offset):
    for ent in entities:
        ent["start"] += offset
        ent["end"] += offset


def assemble_value_line(pairs, sep_style):
    """Собирает строку значений и span-метки без заголовков."""
    entities = []
    parts = []
    pos = 0
    for i, (key, val) in enumerate(pairs):
        if i > 0:
            sep = random.choice(SEP_CHOICES) if sep_style == "mixed" else sep_style
            parts.append(sep)
            pos += len(sep)
        start = pos
        parts.append(val)
        pos += len(val)
        label = "MCC" if key == "mcc" else key.upper()
        entities.append({"start": start, "end": pos, "label": label})
    return "".join(parts), entities


def apply_noise(text, entities):
    """Добавляет шум до/после/внутри строки, сдвигая span-метки."""
    prefix = random.choice(NOISE_PREFIXES) if random.random() < 0.35 else ""
    suffix = random.choice(NOISE_SUFFIXES) if random.random() < 0.25 else ""

    if prefix:
        _shift_entities(entities, len(prefix))
        text = prefix + text

    if random.random() < 0.15 and len(text) > 10:
        mid = random.choice(MID_NOISE)
        insert_at = random.randint(len(text) // 4, 3 * len(text) // 4)
        text = text[:insert_at] + mid + text[insert_at:]
        for ent in entities:
            if ent["start"] >= insert_at:
                _shift_entities([ent], len(mid))

    return text + suffix, entities


def generate_example_with_headers(pairs, sep="\t"):
    """~15% примеров с заголовками — модель учится игнорировать первую строку."""
    HEADER_SYNONYMS = {
        "operation_date": ["Дата операции", "Дата", "Дата транзакции"],
        "posting_date": ["Дата обработки", "Дата платежа"],
        "time": ["Время", "Время операции"],
        "account": ["Счет", "Карта", "Номер счёта"],
        "amount": ["Сумма", "Сумма операции"],
        "currency": ["Валюта", "Валюта операции"],
        "cashback": ["Кэшбэк", "Спасибо", "Бонусы"],
        "category": ["Категория", "Тип операции"],
        "description": ["Описание", "Детали", "Назначение"],
        "mcc": ["MCC", "МСС"],
        "operation_type": ["Тип", "Операция"],
    }
    headers, values = [], []
    for key, val in pairs:
        headers.append(random.choice(HEADER_SYNONYMS.get(key, [key])))
        values.append(val)
    header_line = sep.join(headers)
    value_line = sep.join(values)
    text = header_line + "\n" + value_line
    value_start = len(header_line) + 1
    entities = []
    pos = 0
    for key, val in pairs:
        if pos > 0:
            pos += len(sep)
        start = value_start + pos
        end = start + len(val)
        label = "MCC" if key == "mcc" else key.upper()
        entities.append({"start": start, "end": end, "label": label})
        pos += len(val)
    return text, entities


def generate_example():
    use_fields = {
        "operation_date": True,
        "posting_date": random.random() < 0.5,
        "time": random.random() < 0.6,
        "account": True,
        "amount": True,
        "currency": random.random() < 0.8,
        "cashback": random.random() < 0.4,
        "category": random.random() < 0.9,
        "description": random.random() < 0.85,
        "operation_type": random.random() < 0.55,
    }

    base_date = datetime.date(2026, 1, 1) + datetime.timedelta(days=random.randint(0, 365))
    op_date_str = base_date.strftime(random.choice(["%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"]))
    posting_date_str = (base_date + datetime.timedelta(days=random.randint(0, 3))).strftime(
        random.choice(["%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"])
    )
    time_str = (
        f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
        if random.random() < 0.5
        else f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}"
    )

    is_expense = random.random() < 0.8
    amount = round(random.uniform(10, 15000), 2)
    if not is_expense:
        amount = -amount if random.random() < 0.25 else amount
    amount_str = f"{amount:,.2f}".replace(",", " ") if random.random() < 0.3 else f"{amount:.2f}"
    amount_str = amount_str.replace(".", random.choice([".", ","]))

    currency = "RUB" if random.random() < 0.9 else random.choice(["USD", "EUR"])
    account = f"*{random.randint(1000, 9999)}"
    category_text = random.choice(CATEGORIES) if random.random() < 0.7 else ""
    mcc = random.choice(MCC_CODES) if random.random() < 0.6 else ""
    description = random.choice(DESCRIPTIONS) if use_fields["description"] else ""
    cashback_str = ""
    if use_fields["cashback"]:
        cashback_val = round(random.uniform(1, 500), 2)
        cashback_str = f"{cashback_val:.2f}".replace(".", random.choice([".", ","]))

    pairs = []
    if use_fields["operation_date"]:
        pairs.append(("operation_date", op_date_str))
    if use_fields["posting_date"]:
        pairs.append(("posting_date", posting_date_str))
    if use_fields["time"]:
        pairs.append(("time", time_str))
    if use_fields["account"]:
        pairs.append(("account", account))
    if use_fields["amount"]:
        pairs.append(("amount", amount_str))
    if use_fields["currency"]:
        pairs.append(("currency", currency))
    if use_fields["cashback"] and cashback_str:
        pairs.append(("cashback", cashback_str))
    if category_text:
        pairs.append(("category", category_text))
    if mcc:
        pairs.append(("mcc", mcc))
    if use_fields["description"] and description:
        pairs.append(("description", description))
    if use_fields["operation_type"]:
        type_key = "расход" if is_expense else random.choice(["доход", "перевод"])
        pairs.append(("operation_type", random.choice(OPERATION_TYPE_LABELS[type_key])))

    random.shuffle(pairs)

    if random.random() < 0.15:
        text, entities = generate_example_with_headers(pairs)
    else:
        sep_style = random.choices(
            ["\t", " ", " | ", "", "mixed"],
            weights=[0.25, 0.25, 0.15, 0.15, 0.20],
            k=1,
        )[0]
        text, entities = assemble_value_line(pairs, sep_style)
        text, entities = apply_noise(text, entities)

    return {"text": text, "entities": entities}

In [ ]:
# ========== 2. Генерация датасета ==========
dataset = [generate_example() for _ in range(5000)]
df = pd.DataFrame(dataset)
df.to_csv("receipts_synthetic.csv", index=False)

# ========== 3. Подготовка меток (стандартная BIO-схема) ==========
entity_labels = sorted({ent["label"] for ex in dataset for ent in ex["entities"]})
label_list = ["O"] + [f"B-{l}" for l in entity_labels] + [f"I-{l}" for l in entity_labels]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("Метки:", label_list)

# ========== 4. Токенизация (исправленная) ==========
model_name = "cointegrated/rubert-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_offsets(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        return_offsets_mapping=True,
    )
    labels = []
    for i, entities in enumerate(examples["entities"]):
        offsets = tokenized["offset_mapping"][i]
        label_ids = []
        for idx, (start, end) in enumerate(offsets):
            if start == end:
                label_ids.append(-100)
                continue
            token_label = "O"
            for ent in entities:
                if start >= ent["start"] and end <= ent["end"]:
                    prefix = "B" if start == ent["start"] else "I"
                    token_label = f"{prefix}-{ent['label']}"
                    break
            label_ids.append(label2id[token_label])
        labels.append(label_ids)
    tokenized["labels"] = labels
    del tokenized["offset_mapping"]
    return tokenized

dataset_hf = Dataset.from_list(dataset)
tokenized_dataset = dataset_hf.map(tokenize_and_align_offsets, batched=True, remove_columns=dataset_hf.column_names)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

# ========== 5. Обучение ==========
model = BertForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir="./ner_receipts",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=42,
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for _, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    flat_true = [item for sublist in true_labels for item in sublist]
    flat_pred = [item for sublist in true_predictions for item in sublist]
    return {
        "accuracy": accuracy_score(flat_true, flat_pred),
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./ner_receipts_final")
tokenizer.save_pretrained("./ner_receipts_final")
print("Модель обучена.")

In [ ]:
from datetime import datetime as dt
from transformers import AutoModelForTokenClassification

OPERATION_TYPE_KEYWORDS = {
    "перевод": ["перевод", "p2p"],
    "доход": ["зачисление", "возврат", "поступление"],
    "расход": ["покупка", "списание", "оплата", "платёж", "платеж"],
}
HEADER_MARKERS = [
    "дата", "сумма", "валюта", "время", "категория",
    "описание", "счет", "счёт", "карта", "кэшбэк", "назначение",
]


def _ent_text(text, ent):
    return text[ent["start"]:ent["end"]]


def _value_line_start(text):
    if "\n" not in text:
        return 0
    header, _ = text.split("\n", 1)
    hits = sum(1 for m in HEADER_MARKERS if m in header.lower())
    return len(header) + 1 if hits >= 2 else 0


def _filter_entities(text, entities):
    vs = _value_line_start(text)
    if vs == 0:
        return entities
    return [e for e in entities if e["start"] >= vs]


def _group_entities(entities):
    grouped = {}
    for ent in entities:
        grouped.setdefault(ent["label"].lower(), []).append(ent)
    return grouped


def _is_decimal_tail(s):
    s = s.strip()
    return bool(re.match(r"^[\.,]\d{1,2}$", s))


def _parse_num(s):
    clean = re.sub(r"[^\d,.\-]", "", s).replace(",", ".")
    try:
        return abs(float(clean))
    except ValueError:
        return 0.0


def _resolve_amount_and_cashback(amount_ents, cashback_ents, text):
    amounts = [_ent_text(text, e) for e in amount_ents]
    cashbacks = [_ent_text(text, e) for e in cashback_ents]

    tails = [a for a in amounts if _is_decimal_tail(a)]
    amounts = [a for a in amounts if a not in tails]

    bare_cb_ints = [
        c for c in cashbacks
        if re.fullmatch(r"\d{1,6}", c.strip()) and not re.search(r"[\.,]", c)
    ]
    cashbacks = [c for c in cashbacks if c not in bare_cb_ints]

    for tail in tails:
        tail_norm = tail.replace(",", ".")
        if amounts and bare_cb_ints:
            cb = bare_cb_ints.pop(0)
            cashbacks.append(cb.strip() + tail_norm)
        elif bare_cb_ints:
            cb = bare_cb_ints.pop(0)
            if _parse_num(cb) >= 100:
                amounts.append(cb.strip() + tail_norm)
            else:
                cashbacks.append(cb.strip() + tail_norm)
        elif amounts:
            for i, am in enumerate(amounts):
                if re.fullmatch(r"-?\d+", am.strip().replace(" ", "")):
                    amounts[i] = am.strip() + tail_norm
                    break

    cashbacks.extend(bare_cb_ints)

    candidates = [a for a in amounts if not _is_decimal_tail(a)] or amounts
    amount_raw = max(
        candidates,
        key=lambda a: (_parse_num(a), a.lstrip().startswith("-")),
    ) if candidates else "0"

    cashback_raw = max(cashbacks, key=len) if cashbacks else None
    return amount_raw, cashback_raw


def _pick_longest(entities, text, min_len=3):
    if not entities:
        return None
    texts = [_ent_text(text, e) for e in entities]
    good = [t for t in texts if len(t.strip()) >= min_len]
    pool = good or texts
    return max(pool, key=len)


def detect_operation_type(text, operation_type_texts, amount_raw):
    for ot in operation_type_texts:
        ot_l = ot.lower()
        for op_type in ("перевод", "доход", "расход"):
            if any(k in ot_l for k in OPERATION_TYPE_KEYWORDS[op_type]):
                return op_type

    text_lower = text.lower()
    for op_type in ("перевод", "доход", "расход"):
        if any(k in text_lower for k in OPERATION_TYPE_KEYWORDS[op_type]):
            return op_type

    if amount_raw and amount_raw.lstrip().startswith("-"):
        return "расход"

    return "доход"


def parse_receipt(text, debug=False, model_inf=None, tokenizer_inf=None):
    if tokenizer_inf is None:
        tokenizer_inf = AutoTokenizer.from_pretrained("./ner_receipts_final")
    if model_inf is None:
        model_inf = AutoModelForTokenClassification.from_pretrained("./ner_receipts_final")
        model_inf.eval()

    inputs = tokenizer_inf(text, return_offsets_mapping=True, return_tensors="pt", truncation=True)
    offsets = inputs.pop("offset_mapping").squeeze(0).tolist()
    with torch.no_grad():
        outputs = model_inf(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2).squeeze(0).tolist()

    entities = []
    current_entity = None
    for pred_id, (start, end) in zip(predictions, offsets):
        if start == end:
            continue
        label = model_inf.config.id2label[pred_id]
        if label == "O":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {"label": label[2:], "start": start, "end": end}
        elif label.startswith("I-") and current_entity and current_entity["label"] == label[2:]:
            current_entity["end"] = end
        else:
            if current_entity:
                entities.append(current_entity)
            etype = label[2:] if label.startswith("I-") else label
            current_entity = {"label": etype, "start": start, "end": end}
    if current_entity:
        entities.append(current_entity)

    entities = _filter_entities(text, entities)

    if debug:
        print("=== Сущности ===")
        for ent in entities:
            print(f"  {ent['label']}: '{_ent_text(text, ent)}'")

    grouped = _group_entities(entities)

    date_str = _pick_longest(grouped.get("operation_date", []), text) or ""
    time_str = _pick_longest(grouped.get("time", []), text) or ""
    op_date = None
    if date_str:
        for fmt in ["%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"]:
            try:
                op_date = dt.strptime(date_str, fmt)
                break
            except ValueError:
                continue
    if op_date and time_str:
        for tf in ["%H:%M:%S", "%H:%M"]:
            try:
                t = dt.strptime(time_str, tf).time()
                op_date = dt.combine(op_date.date(), t)
                break
            except ValueError:
                continue

    amount_raw, cashback_raw = _resolve_amount_and_cashback(
        grouped.get("amount", []),
        grouped.get("cashback", []),
        text,
    )
    operation_type_texts = [_ent_text(text, e) for e in grouped.get("operation_type", [])]

    result = {
        "operation_date": op_date.isoformat() if op_date else None,
        "amount": _parse_num(amount_raw),
        "currency": _pick_longest(grouped.get("currency", []), text, min_len=3) or "RUB",
        "operation_type": detect_operation_type(text, operation_type_texts, amount_raw),
        "account": (_pick_longest(grouped.get("account", []), text, min_len=4) or "").replace("*", ""),
        "category": _pick_longest(grouped.get("category", []), text) or _pick_longest(grouped.get("mcc", []), text) or "",
        "description": _pick_longest(grouped.get("description", []), text) or "",
        "cashback": cashback_raw,
    }

    return {k: result.get(k) for k in [
        "operation_date", "amount", "currency", "operation_type",
        "account", "category", "description", "cashback",
    ]}


TEST_CASES = [
    {
        "name": "Табы, минус, расход",
        "text": "26.05.2026\t20:50:54\t-329,97\tRUB\t*0367\tСупермаркеты\tМагнит 3\t12.50",
        "expected_type": "расход",
        "expected_amount": 329.97,
        "expected_cashback": "12.50",
    },
    {
        "name": "Метка Покупка",
        "text": "Покупка 26.05.2026 20:50:54 -329,97 RUB *0367 Супермаркеты Магнит 3",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Зачисление (доход)",
        "text": "Зачисление 15.03.2026 14:22 15000.00 RUB *5958 Поступление зарплаты",
        "expected_type": "доход",
        "expected_amount": 15000.0,
    },
    {
        "name": "Перевод P2P",
        "text": "Перевод на карту 01.02.2026 -5000,00 RUB *1234 Перевод на карту",
        "expected_type": "перевод",
        "expected_amount": 5000.0,
    },
    {
        "name": "Слипшийся текст",
        "text": "26.05.202620:50:54-329,97RUB*0367СупермаркетыМагнит 3",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Шум до и после",
        "text": "Справка по операции 26.05.2026 | 20:50:54 | -329,97 | RUB | *0367 | Супермаркеты | Магнит 3 Обработано банком.",
        "expected_type": "расход",
        "expected_amount": 329.97,
    },
    {
        "name": "Положительная сумма без минуса",
        "text": "Оплата 10.01.2026 12:00 1500.00 RUB *7777 Рестораны Макдоналдс",
        "expected_type": "расход",
        "expected_amount": 1500.0,
    },
    {
        "name": "EUR, дата через слэш",
        "text": "20/01/2026 04:30:37 8070.42 EUR *5958 Домашние животные МТС",
        "expected_type": "доход",
        "expected_amount": 8070.42,
    },
    {
        "name": "Старый формат с заголовками (robustness)",
        "text": "Дата операции\tВремя\tСумма\tВалюта\tКарта\tКатегория\tОписание\tКэшбэк\n26.05.2026\t20:50:54\t-329,97\tRUB\t*0367\tСупермаркеты\tМагнит 3\t12.50",
        "expected_type": "расход",
        "expected_amount": 329.97,
        "expected_cashback": "12.50",
    },
]

print("=== Тесты инференса ===\n")
_tokenizer = AutoTokenizer.from_pretrained("./ner_receipts_final")
_model = AutoModelForTokenClassification.from_pretrained("./ner_receipts_final")
_model.eval()

for case in TEST_CASES:
    print(f"--- {case['name']} ---")
    parsed = parse_receipt(case["text"], debug=True, model_inf=_model, tokenizer_inf=_tokenizer)
    type_ok = parsed["operation_type"] == case["expected_type"]
    amount_ok = abs(parsed["amount"] - case["expected_amount"]) < 0.01 if "expected_amount" in case else True
    cb_ok = True
    if "expected_cashback" in case:
        cb_ok = parsed["cashback"] == case["expected_cashback"]
    print(parsed)
    print(
        f"тип: {'OK' if type_ok else 'FAIL'} ({case['expected_type']} / {parsed['operation_type']}) | "
        f"сумма: {'OK' if amount_ok else 'FAIL'} ({case.get('expected_amount', '-')} / {parsed['amount']}) | "
        f"кэшбэк: {'OK' if cb_ok else 'FAIL'}"
    )
    print()